# Ngọc Huyền TTS

> **GPU:** T4 16GB (tự động) · **Thời gian:** ~3 phút lần đầu, ~30s/lần sau

## Hướng dẫn

1. **Runtime** → **Run all** (Ctrl+F9)
2. Nhập văn bản → Click **Tạo giọng nói**
3. Audio tự động tải về máy (MP3) + hiển thị trên giao diện
4. Bấm **📦 Tải tất cả audio** để tải zip toàn bộ

### Tối ưu đã áp dụng
- ⚡ Speed 1.25x (giọng đọc nhanh hơn 25%)
- ⚡ `num_step=16` (inference nhanh ~2x so với 32)
- ⚡ FlashInfer GPU kernel acceleration
- 📥 Tự động tải audio MP3 về máy sau mỗi lần tạo
- 🎵 Hiển thị audio mới nhất trên giao diện
- 📦 Nút tải toàn bộ audio đã tạo (zip)

> Powered by OmniVoice (Apache 2.0) — github.com/k2-fsa/OmniVoice


In [ ]:
# ==========================================
# Cell 1: Cài đặt dependencies
# ==========================================
print('🔧 Đang cài đặt dependencies (~1-2 phút)...')

# Core: OmniVoice + Gradio + numpy compat
!pip install -q omnivoice gradio "numpy<2.1" "requests==2.32.4"

# Audio processing: pydub cho xuất MP3
!pip install -q pydub soundfile

# ffmpeg cho pydub MP3 export
!apt-get -qq install -y ffmpeg > /dev/null 2>&1

# FlashInfer acceleration (~2-3x speedup)
import subprocess, sys
try:
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '-q', 'flashinfer-python==0.6'],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )
    print('✅ FlashInfer installed')
except Exception:
    print('⚠️ FlashInfer không cài được (vẫn chạy OK, chỉ chậm hơn một chút)')

# Tải giọng mẫu Ngọc Huyền (self-hosted — repo của mình)
!wget -q https://raw.githubusercontent.com/vinh-nd2002/tts/main/ngoc-huyen.mp3 -O voice_sample.mp3

print('✅ Cài đặt hoàn tất!')
print('✅ Đã tải voice sample Ngọc Huyền!')

In [ ]:
# ==========================================
# Cell 2: Khởi động model + Giao diện TTS
# ==========================================
print('🚀 Đang khởi động OmniVoice... (lần đầu ~3-5 phút, lần sau ~30 giây)')

import os, sys, time, re, datetime, shutil, logging
import numpy as np
import torch
import soundfile as sf

logger = logging.getLogger('omnivoice')
logging.basicConfig(level=logging.INFO)

from omnivoice import OmniVoice, OmniVoiceGenerationConfig
import gradio as gr

def get_best_device():
    if torch.cuda.is_available():
        return 'cuda'
    return 'cpu'

# Wait for GPU
for _ in range(30):
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f'🖥️ GPU: {gpu_name} ({gpu_mem:.1f} GB)')
        break
    time.sleep(1)
else:
    print('❌ GPU not available — Runtime > Change runtime type > T4 GPU')

# ── Load Model ─────────────────────────────────────────────────────
DEVICE = get_best_device()
logger.info(f'Loading OmniVoice on {DEVICE}...')
model = OmniVoice.from_pretrained(
    'k2-fsa/OmniVoice', device_map=DEVICE, dtype=torch.float16, load_asr=True
)
SAMPLING_RATE = model.sampling_rate
logger.info(f'✅ Model ready — SR: {SAMPLING_RATE}Hz')

# ── Voice Clone Prompt ─────────────────────────────────────────────
logger.info('Creating VoiceClonePrompt...')
VOICE_PROMPT = model.create_voice_clone_prompt(ref_audio='voice_sample.mp3')
logger.info('✅ Voice prompt ready — Ngọc Huyền')

# ── Generation Config (OPTIMIZED) ──────────────────────────────────
GEN_CFG = OmniVoiceGenerationConfig(
    num_step=16,
    guidance_scale=1.8,
    denoise=True,
    preprocess_prompt=True,
    postprocess_output=True,
    position_temperature=5.0,
    class_temperature=0.2,
    pad_duration=0.1,
    fade_duration=0.1,
)

SPEAK_SPEED = 1.25

# ── Output Directory ──────────────────────────────────────────────
OUTPUT_DIR = '/content/audio_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

audio_history = []

# ── Auto Download Helper ──────────────────────────────────────────
try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def get_system_stats():
    """Lấy thông tin CPU, RAM, GPU hiện tại."""
    import psutil
    stats = {}
    stats['cpu'] = psutil.cpu_percent(interval=0.1)
    mem = psutil.virtual_memory()
    stats['ram_used'] = mem.used / 1024**3
    stats['ram_total'] = mem.total / 1024**3
    stats['ram_pct'] = mem.percent

    if torch.cuda.is_available():
        stats['gpu_mem_used'] = torch.cuda.memory_allocated() / 1024**3
        stats['gpu_mem_total'] = torch.cuda.get_device_properties(0).total_memory / 1024**3
        stats['gpu_mem_pct'] = (stats['gpu_mem_used'] / stats['gpu_mem_total']) * 100
        stats['gpu_name'] = torch.cuda.get_device_name(0)
    else:
        stats['gpu_mem_used'] = 0
        stats['gpu_mem_total'] = 0
        stats['gpu_mem_pct'] = 0
        stats['gpu_name'] = 'N/A'

    return stats

def format_time(seconds):
    """Format seconds to human readable."""
    if seconds < 60:
        return f"{seconds:.1f}s"
    minutes = int(seconds // 60)
    secs = seconds % 60
    return f"{minutes}m {secs:.0f}s"

def estimate_duration(text, speed=1.25):
    """Ước lượng thời gian audio output (giây) dựa trên số từ."""
    words = len(text.split())
    # Tiếng Việt ~300 từ/phút ở tốc độ bình thường
    minutes = words / (300 * speed)
    return minutes * 60

def estimate_processing_time(text):
    """Ước lượng thời gian xử lý (giây) dựa trên số ký tự."""
    chars = len(text)
    # Benchmark: ~100 ký tự/giây trên T4 GPU với num_step=16
    return max(5, chars / 100)

def save_audio_mp3(audio_np, sr):
    """Lưu audio thành MP3 + tự động tải về máy."""
    from pydub import AudioSegment
    ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    wav_path = os.path.join(OUTPUT_DIR, f'ngoc_huyen_{ts}.wav')
    mp3_path = os.path.join(OUTPUT_DIR, f'ngoc_huyen_{ts}.mp3')

    sf.write(wav_path, audio_np, sr)
    AudioSegment.from_wav(wav_path).export(mp3_path, format='mp3', bitrate='192k')
    os.remove(wav_path)

    audio_history.append(mp3_path)
    logger.info(f'💾 Saved: {mp3_path}')

    if IN_COLAB:
        try:
            colab_files.download(mp3_path)
            logger.info(f'📥 Auto download: {os.path.basename(mp3_path)}')
        except Exception as e:
            logger.warning(f'Auto download failed: {e}')

    return mp3_path

# ── Generate Function with Progress ──────────────────────────────
@torch.inference_mode()
def generate_voice(text: str, progress=gr.Progress(track_tqdm=True)):
    """Generate voice with live progress updates."""
    text = text.strip()
    if not text:
        yield None, None, '❌ Vui lòng nhập văn bản!'
        return

    start_time = time.time()
    start_dt = datetime.datetime.now().strftime('%H:%M:%S %d-%m-%Y')

    # Stats
    word_count = len(text.split())
    char_count = len(text)
    est_audio = estimate_duration(text)
    est_process = estimate_processing_time(text)

    # Split paragraphs
    paragraphs = [p.strip() for p in re.split(r'\n\s*\n', text) if p.strip()]
    total_paragraphs = len(paragraphs)

    # ── PHASE 1: Show estimation ──
    stats = get_system_stats()
    init_status = (
        f'⏳ ĐANG XỬ LÝ...\n'
        f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n'
        f'📊 Input: {word_count} từ · {char_count} ký tự · {total_paragraphs} đoạn\n'
        f'⏱️ Ước lượng: ~{format_time(est_process)} xử lý → ~{format_time(est_audio)} audio\n'
        f'🕐 Bắt đầu: {start_dt}\n'
        f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n'
        f'🖥️ GPU: {stats["gpu_name"]} ({stats["gpu_mem_used"]:.1f}/{stats["gpu_mem_total"]:.1f} GB · {stats["gpu_mem_pct"]:.0f}%)\n'
        f'💾 RAM: {stats["ram_used"]:.1f}/{stats["ram_total"]:.1f} GB · {stats["ram_pct"]:.0f}%\n'
        f'🔧 CPU: {stats["cpu"]:.0f}%\n'
        f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n'
        f'▶️ Đang tạo giọng nói... 0/{total_paragraphs} đoạn'
    )
    yield None, None, init_status

    # ── PHASE 2: Generate each paragraph ──
    audios = []
    for i, p in enumerate(paragraphs):
        para_start = time.time()
        elapsed = time.time() - start_time

        # Update progress
        pct = int((i / total_paragraphs) * 100)
        progress(i / total_paragraphs, desc=f'Đoạn {i+1}/{total_paragraphs}')

        # Generate
        a = model.generate(
            text=p,
            voice_clone_prompt=VOICE_PROMPT,
            language='vi',
            speed=SPEAK_SPEED,
            generation_config=GEN_CFG
        )[0]
        audios.append(a)

        # Add silence between paragraphs
        if i < total_paragraphs - 1:
            audios.append(np.zeros(int(SAMPLING_RATE * 0.3)))

        # Update stats after each paragraph
        para_elapsed = time.time() - para_start
        total_elapsed = time.time() - start_time

        # Calculate speed and ETA
        chars_done = sum(len(paragraphs[j]) for j in range(i + 1))
        chars_remaining = sum(len(paragraphs[j]) for j in range(i + 1, total_paragraphs))
        speed_cps = chars_done / total_elapsed if total_elapsed > 0 else 0
        eta = chars_remaining / speed_cps if speed_cps > 0 else 0

        pct_done = int(((i + 1) / total_paragraphs) * 100)
        bar_filled = int(pct_done / 5)
        bar = '█' * bar_filled + '░' * (20 - bar_filled)

        stats = get_system_stats()
        progress_status = (
            f'⏳ ĐANG XỬ LÝ... [{bar}] {pct_done}%\n'
            f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n'
            f'📊 Input: {word_count} từ · {char_count} ký tự · {total_paragraphs} đoạn\n'
            f'🕐 Bắt đầu: {start_dt}\n'
            f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n'
            f'✅ Đoạn {i+1}/{total_paragraphs} xong ({para_elapsed:.1f}s)\n'
            f'⚡ Tốc độ: {speed_cps:.0f} ký tự/s\n'
            f'⏱️ Đã chạy: {format_time(total_elapsed)} · Còn lại: ~{format_time(eta)}\n'
            f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n'
            f'🖥️ GPU: {stats["gpu_mem_used"]:.1f}/{stats["gpu_mem_total"]:.1f} GB · {stats["gpu_mem_pct"]:.0f}%\n'
            f'💾 RAM: {stats["ram_used"]:.1f}/{stats["ram_total"]:.1f} GB · {stats["ram_pct"]:.0f}%\n'
            f'🔧 CPU: {stats["cpu"]:.0f}%'
        )
        yield None, None, progress_status

    # ── PHASE 3: Finalize ──
    progress(0.95, desc='Đang lưu MP3...')
    audio = np.concatenate(audios)
    elapsed = time.time() - start_time
    duration = len(audio) / SAMPLING_RATE

    # Save MP3 + auto download
    mp3_path = save_audio_mp3(audio, SAMPLING_RATE)

    # Prepare audio for Gradio player
    waveform = (audio * 32767).astype(np.int16)

    end_dt = datetime.datetime.now().strftime('%H:%M:%S %d-%m-%Y')
    speed_cps = char_count / elapsed if elapsed > 0 else 0
    rtf = elapsed / duration if duration > 0 else 0

    stats = get_system_stats()
    final_status = (
        f'✅ HOÀN THÀNH! [████████████████████] 100%\n'
        f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n'
        f'📊 Input: {word_count} từ · {char_count} ký tự · {total_paragraphs} đoạn\n'
        f'🎵 Audio: {format_time(duration)} · MP3 192kbps\n'
        f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n'
        f'🕐 Bắt đầu:  {start_dt}\n'
        f'🕐 Kết thúc: {end_dt}\n'
        f'⏱️ Tổng xử lý: {format_time(elapsed)}\n'
        f'⚡ Tốc độ: {speed_cps:.0f} ký tự/s · RTF: {rtf:.2f}x\n'
        f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n'
        f'🖥️ GPU: {stats["gpu_mem_used"]:.1f}/{stats["gpu_mem_total"]:.1f} GB\n'
        f'💾 RAM: {stats["ram_used"]:.1f}/{stats["ram_total"]:.1f} GB\n'
        f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n'
        f'📥 {os.path.basename(mp3_path)}\n'
        f'📁 Tổng audio đã tạo: {len(audio_history)} file'
    )

    progress(1.0, desc='Hoàn thành!')
    yield (SAMPLING_RATE, waveform), mp3_path, final_status

# ── Download All Function ─────────────────────────────────────────
def download_all_audio():
    """Nén tất cả audio → zip → tải về."""
    if not audio_history:
        return None, '❌ Chưa có audio nào để tải!'

    zip_path = '/content/all_audio'
    shutil.make_archive(zip_path, 'zip', OUTPUT_DIR)
    zip_file = f'{zip_path}.zip'

    if IN_COLAB:
        try:
            colab_files.download(zip_file)
        except Exception:
            pass

    status = f'📦 Đã nén {len(audio_history)} file audio → all_audio.zip'
    return zip_file, status

# ── Gradio UI ──────────────────────────────────────────────────────
print('🎨 Khởi động giao diện OmniVoice TTS — Ngọc Huyền...')
gr.close_all()

CSS = """
.gradio-container {
    max-width: 780px !important;
    margin: 0 auto !important;
    padding: 16px !important;
}
footer { display: none !important; }
.status-box textarea {
    font-family: 'Courier New', monospace !important;
    font-size: 13px !important;
    line-height: 1.5 !important;
    background: linear-gradient(135deg, #0f0c29, #302b63, #24243e) !important;
    color: #e0e0e0 !important;
    border: 1px solid #4f46e5 !important;
    border-radius: 12px !important;
    padding: 16px !important;
}
"""

THEME = gr.themes.Soft(primary_hue='indigo')

with gr.Blocks(title='OmniVoice TTS — Ngọc Huyền') as demo:
    gr.Markdown(
        '# 🎙️ OmniVoice TTS\n'
        '**Ngọc Huyền** · Giọng nữ thanh niên, tự nhiên, truyền cảm\n\n'
        '`Speed: 1.25x` · `Steps: 16` · `Output: MP3 192kbps` · `Auto Download: ON`'
    )

    with gr.Row():
        text_input = gr.Textbox(
            label='📝 Nhập văn bản',
            lines=6,
            placeholder='Nhập văn bản bạn muốn chuyển thành giọng nói...\n\nTách đoạn bằng 2 dòng trống để có khoảng nghỉ giữa các đoạn.'
        )

    with gr.Row():
        btn_generate = gr.Button('🎤 Tạo giọng nói', variant='primary', scale=3)
        btn_download_all = gr.Button('📦 Tải tất cả audio', variant='secondary', scale=1)

    # Status display — live progress
    status_text = gr.Textbox(
        label='📊 Trạng thái',
        interactive=False,
        lines=12,
        value='🟢 Sẵn sàng! Nhập văn bản và bấm Tạo giọng nói.',
        elem_classes=['status-box']
    )

    # Audio player
    audio_output = gr.Audio(label='🎵 Audio mới nhất', type='numpy')

    # Download file
    file_output = gr.File(label='📥 File MP3 mới nhất (click để tải)')

    # Zip file
    zip_output = gr.File(label='📦 Zip tất cả audio', visible=False)

    # Wire up events
    btn_generate.click(
        fn=generate_voice,
        inputs=[text_input],
        outputs=[audio_output, file_output, status_text],
        concurrency_limit=1
    )

    btn_download_all.click(
        fn=download_all_audio,
        inputs=[],
        outputs=[zip_output, status_text]
    ).then(
        fn=lambda: gr.update(visible=True),
        outputs=[zip_output]
    )

# Launch
demo.launch(
    server_name='0.0.0.0',
    share=True,
    theme=THEME,
    css=CSS,
    debug=True
)
